# K513 · Week 4, Session 1
## Introduction to regression — your first model

Last Thursday you split a table into training and test data without having anything to test.
Today you build the thing.

One question runs through the whole notebook:

> **How much is this tract worth — and how much better than the average is that?**

Three sections:

| Section | The question |
|---|---|
| 1 | What would you predict with no model at all? |
| 2 | Can twelve columns beat that baseline — and what is the model actually saying? |
| 3 | What happens when a column isn't a number? |

By the end you will have a model, a number in dollars, and a sentence you could put in front of
somebody who does not code. The sentence is the part that gets graded.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

The specific trap in this session is the coefficient. Ask an AI what a coefficient means and you
will get a correct, generic sentence: *a one-unit increase in X is associated with a change of b in
Y.* It does not know what one unit of your column is, whether one unit is a thing that can happen,
or what your target is measured in. Every wrong answer people give about regression is a correct
generic sentence applied to a column nobody looked at.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.

---
## 0 · Setup

Two libraries you know and one that is new. `scikit-learn` is the machine-learning library — it
holds the models, the tools for preparing data, and the scoring functions. It is the only modeling
library this course uses.

Run both cells; neither shows anything.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option('display.precision', 3)

In [ ]:
BOSTON_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/BostonHousing.csv"
TOYOTA_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/Used_Toyota.csv"

boston_df = pd.read_csv(BOSTON_URL)
boston_df.head()

---
## 1 · What would you predict with no model at all?

Before any code, the four questions from last Thursday:

1. **Unit of analysis** — one row is one census tract in the Boston area. Not one house.
2. **Target** — `MEDV`, the median home value in that tract, in **thousands of dollars**.
3. **Task** — `MEDV` is a number, so this is regression.
4. **Baseline** — predict the same value for every tract: the average.

That fourth one is what this section builds. Nothing can be called a good model until something
else is worse.

### Split first

This is the order that matters, and it will matter more in section 3. The test rows are held back
so that we can ask, honestly, how the model does on data it has never seen.

`random_state=42` fixes *which* rows go where, so your split is the same as everyone else's. Without
it you get a different 379 rows every time you run the cell.

In [ ]:
X = boston_df.drop(columns=['MEDV'])
y = boston_df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print("training rows:", X_train.shape[0])
print("test rows:    ", X_test.shape[0])

### The baseline

The baseline model has no predictors and nothing to fit. It says the same thing about every tract:
the average.

Note that it is the **training** average. Using the average of all 506 rows would mean the test rows
had already influenced the prediction — a small version of exactly the mistake section 3 is about.

In [ ]:
baseline = y_train.mean()
print(f"the baseline: ${baseline * 1000:,.0f}")

### How wrong is it?

Two ways to score it, both in dollars, and they are **not** the same number:

- **RMSE** — square every error, average, take the square root. This is the one to report. It is
  what least squares actually minimizes, so it is the quantity every model in this notebook is
  built to make small.
- **MAE** — the plain average size of an error — the typical miss. This is the one you can say
  out loud: *a tract's median value is typically off by about six thousand dollars.*

RMSE is always at least as large as MAE, and the gap between them says how **uneven** the errors
are. You will see what causes that gap in section 2.

Remember `MEDV` is in thousands, so multiply by 1,000 to get a number you can say out loud.

In [ ]:
baseline_pred = np.full(len(y_test), baseline)

rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
mae  = mean_absolute_error(y_test, baseline_pred)

print(f"RMSE: ${rmse * 1000:,.0f}   <- the number to beat")
print(f"MAE : ${mae  * 1000:,.0f}")

### ✏️ Now You Try · 1

Work with the person beside you.

**a)** The three cells above did the work. Read them again and answer in words, not code:
why did we use `y_train.mean()` rather than `y.mean()` in baseline?

*Your answer:*

**b)** Fill in the blanks to score the baseline with R² as well. R² asks what share of the
baseline's error a model removes.

You should get about **−0.03** — essentially zero, which is the whole point of a baseline. If you
are wondering about the minus sign: R² scores against the *test set's own* average, and we predicted
the *training* average. Predicting a number from somewhere else is a hair worse than predicting the
average of the rows you are scoring.

In [ ]:
from sklearn.metrics import r2_score

r2_baseline = r2_score(____, ____)
print(f"R² of the baseline: {r2_baseline:.3f}")

**c)** Which of RMSE and MAE calculated above is larger? Will that always be true, whatever the data? Answer in
words, not code.

*Your answer:*

**d)** In one sentence, in your own words: what does that **MAE** mean, in dollars, to
somebody deciding which neighborhood to buy in?

Use MAE and not RMSE for this one. *Typically off by about \$6,223* is a true sentence; the same
sentence with the RMSE in it is not, and section 2 shows you why.

*Your sentence:*

> **\$8,501 is the number to beat.** Write it somewhere you can see it. Everything in
> section 2 is measured against it.

---
## 2 · From one column to twelve

Two models, and they are the same two lines of code — the only difference is how many columns go in.

`LinearRegression()` finds the straight-line combination of the columns that makes MSE (total squared
error) smallest — that is what **least squares** means, and it is the whole of what this line does.

### One column first — the line from earlier

This is the line you looked at against `LSTAT`, actually fitted: the one the dashed error segments
were measured against. Two lines to build it, two to score it.

In [ ]:
lstat_only = LinearRegression().fit(X_train[['LSTAT']], y_train)
pred_lstat = lstat_only.predict(X_test[['LSTAT']])

rmse_lstat = np.sqrt(mean_squared_error(y_test, pred_lstat))
print(f"LSTAT only        ${rmse_lstat * 1000:,.0f}")

### Now all twelve

Same two lines, a wider `X`. `X_train` already holds all twelve columns, so there is nothing to
redefine — the double brackets above were selecting one column out of the twelve.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

### Score it on both sets

Always both. The training score tells you whether the model learned anything at all; the test score
tells you whether what it learned generalizes. A model that scores well on training and badly on
test has **overfitted** — that is Thursday's subject, and these two numbers are your reference
point for what *not* overfitting looks like.

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

print(f"{'':10}{'train':>10}{'test':>10}")
print(f"{'R²':10}{model.score(X_train, y_train):>10.3f}{model.score(X_test, y_test):>10.3f}")
print(f"{'RMSE $':10}"
      f"{np.sqrt(mean_squared_error(y_train, y_train_pred)) * 1000:>10,.0f}"
      f"{np.sqrt(mean_squared_error(y_test,  y_test_pred))  * 1000:>10,.0f}")
print(f"{'MAE  $':10}"
      f"{mean_absolute_error(y_train, y_train_pred) * 1000:>10,.0f}"
      f"{mean_absolute_error(y_test,  y_test_pred)  * 1000:>10,.0f}")

### The scoreboard

All three numbers, together. This is the chart from class, and every bar on it is now something you
have produced yourself.

One column closes about a third of the gap and the other twelve close the rest. More columns help,
but not in proportion to how many you add.

In [ ]:
print(f"baseline          ${np.sqrt(mean_squared_error(y_test, baseline_pred)) * 1000:>7,.0f}")
print(f"LSTAT only        ${rmse_lstat * 1000:>7,.0f}")
print(f"twelve columns    ${np.sqrt(mean_squared_error(y_test, y_test_pred)) * 1000:>7,.0f}")

### What the model is actually saying

This is the part that ends up in a memo. Everything above is machinery.

A coefficient answers one question: **holding every other column constant, what does one more unit
of this column do to the prediction?** Because `MEDV` is in thousands of dollars, a coefficient of
`4.197` means about **\$4,197**.

In [ ]:
coefficients = pd.DataFrame({
    'column':      X.columns,
    'coefficient': model.coef_,
    'dollars':     model.coef_ * 1000,
})
coefficients.sort_values('coefficient', ascending=False)

### ✏️ Now You Try · 2

**a)** Your three RMSE figures should read \$8,501 for the baseline, \$5,959 for `LSTAT`
alone, and \$4,523 for all twelve columns — the scoreboard from class.

Did your model beat the baseline? By how much, in dollars — and does the answer change if you compare
on MAE instead?

*Your answer:*

**b)** Write one sentence, in dollars, about `RM` — the average number of rooms per dwelling in
the tract. Start it with *"Holding everything else constant, ..."*

*Your sentence:*

**c)** Now do the same for `NOX`, the nitric-oxide concentration. Run the cell below **first**.

This one is a trap, and the cell shows you why.

In [ ]:
print(boston_df['NOX'].describe()[['min', 'max']])

*Your sentence about `NOX`:*

*(Hint: the coefficient is about −17.3, but look at that range. Is a one-unit increase in `NOX`
something that can happen to a Boston tract? Quote a move that can.)*

**d)** Two weeks ago you found that `RAD` correlates **−0.382** with `MEDV`. Run the cell below
and compare that with `RAD`'s coefficient in your table.

They have opposite signs. Both numbers are correct. In one or two sentences: why?

In [ ]:
print("correlation of RAD with MEDV:", round(boston_df['RAD'].corr(boston_df['MEDV']), 3))
print("correlation of RAD with TAX: ", round(boston_df['RAD'].corr(boston_df['TAX']), 3))

*Your answer:*

### Argue with the model

Two things are wrong with this model and neither of them shows up in R², MAE or RMSE. You find them
by looking at a picture.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([-10, 55], [-10, 55], 'k--', lw=1)
plt.axhline(0, color='gray', lw=0.8)
plt.xlabel("the tract's actual median value ($000)")
plt.ylabel("model's prediction ($000)")
plt.show()

print("lowest prediction the model made: $%,.0f" .replace('%,', '%') % (y_test_pred.min() * 1000))
print("tracts in the test set that sold at exactly $50.0k:", (y_test == 50).sum())

The dashed line is where a perfect model would sit.

**The model quotes a negative price.** A linear model will happily put a tract's median value
below zero, because nothing inside it knows what a house is.

**`MEDV` was capped at 50.** Anything worth more than \$50,000 was written down as exactly 50 when
the data was collected. The ceiling is in the data and the model cannot see it — which is why the
points on the right-hand edge sit so far below the line.

**And this is the gap from section 1.** That single capped tract carries about a quarter of the
model's total squared error, while contributing only about 7% of its total absolute error. Squaring
is what makes one bad row count that heavily — which is exactly why RMSE (\$4,523) sits so far
above the typical miss (\$2,974). Run the cell below to see it.

Neither of these is a bug you fix today. Knowing they are there is the deliverable.

In [ ]:
abs_err = np.abs(y_test - y_test_pred) * 1000
worst   = np.sort(abs_err)[::-1]

print(f"RMSE / MAE ratio: {np.sqrt((abs_err**2).mean()) / abs_err.mean():.2f}")
print(f"  (about 1.25 would mean the errors look normal; higher means a few dominate)")
print()
print(f"median miss: ${np.median(abs_err):,.0f}")
print(f"largest miss: ${worst[0]:,.0f}")
print(f"share of total SQUARED error from the worst 1 tract: "
      f"{worst[0]**2 / (abs_err**2).sum():.0%}")
print(f"share of total ABSOLUTE error from the worst 1 tract: "
      f"{worst[0] / abs_err.sum():.0%}")

---
## 3 · When a column isn't a number

This week's homework uses `Used_Toyota.csv` — 1,416 used cars, and the target is `Price`.

In [ ]:
toyota_df = pd.read_csv(TOYOTA_URL)
toyota_df.head(3)

### Run section 2's code on it

Same five lines. Nothing has changed except the table.

**This cell is supposed to fail.** Read the error message before you do anything else — it says
exactly what is wrong, in plain English.

Because it raises, *Runtime → Run all* will stop here. That is expected. Read the message, then
carry on running the cells below one at a time.

In [ ]:
X_t = toyota_df.drop(columns=['Price'])
y_t = toyota_df['Price']

X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(
        X_t, y_t, test_size=0.25, random_state=42)

LinearRegression().fit(X_t_train, y_t_train)

```
ValueError: could not convert string to float: '2/3-Doors'
```

**A model can only do arithmetic.** It multiplies every column by a coefficient and adds up the
results. There is no number that a door configuration can be multiplied by.

So every column going into a model has to be a number — and *how* we turn text into numbers changes
what the model can learn.

### One column becomes two

`Num_Doors` has three values. `OneHotEncoder(drop='first')` turns that one column of text into
**two** columns of 0 and 1:

| `Num_Doors` | `4/5-Doors` | `Stationwagen` |
|---|---|---|
| `2/3-Doors` | 0 | 0 |
| `4/5-Doors` | 1 | 0 |
| `Stationwagen` | 0 | 1 |

The value left out — `2/3-Doors` — becomes the **reference level**: the row where both new columns
are zero. Every coefficient is then read as a difference *from* it.

Leaving one out is what `drop='first'` does, and it is not optional. Three columns for three
categories carries the same information twice, and the model then cannot tell which one deserves
the credit.

### Two tools

- **`ColumnTransformer`** — do different things to different columns. Each entry is a name, a tool,
  and the list of columns it applies to. `'passthrough'` means leave these alone.
- **`Pipeline`** — do these steps in this order, every single time. This is what makes it hard to
  accidentally prepare the test set using the test set.

### Building it together

We run these four cells in class. This week's homework asks you to build the same thing yourself,
on a wider set of columns — so follow what each line is doing rather than copying it down.

**Which of these columns are categories?** Four of them are, and only two of those look
like it.

`Engine_Size` stores 1.3, 1.6, 2.0 — but is a 2.0 engine worth exactly twice a 1.0? If not, it is a
label, not a quantity. `Mfr_Guarantee` and `Power_Steering` are labels too: 0 and 1 stand for no and
yes, and 1 is not twice 0.

Encoding those last two changes nothing — a two-level category, once encoded, *is* a 0/1 column, so
they arrived already encoded. Only `Age_Month` and `KM` are real quantities. This is the first
question on the homework.

In [ ]:
categorical = ['Num_Doors', 'Engine_Size', 'Mfr_Guarantee', 'Power_Steering']
continuous     = ['Age_Month', 'KM']

print(categorical)
print(continuous)

**Build the `ColumnTransformer` and the `Pipeline`, then fit.**

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first'), categorical),
    ('num', 'passthrough',               continuous)])

toyota_model = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor',    LinearRegression())])

toyota_model.fit(X_t_train, y_t_train)

**Score it against the baseline** for this dataset — predicting the average price for every
car. Report RMSE, as we did for Boston.

In [ ]:
t_train_pred = toyota_model.predict(X_t_train)
t_test_pred  = toyota_model.predict(X_t_test)

t_baseline = np.full(len(y_t_test), y_t_train.mean())

print(f"baseline RMSE: ${np.sqrt(mean_squared_error(y_t_test, t_baseline)):>8,.0f}")
print(f"model    RMSE: ${np.sqrt(mean_squared_error(y_t_test, t_test_pred)):>8,.0f}")
print(f"model    MAE:  ${mean_absolute_error(y_t_test, t_test_pred):>8,.0f}")
print()
print(f"R² train: {toyota_model.score(X_t_train, y_t_train):.3f}")
print(f"R² test:  {toyota_model.score(X_t_test,  y_t_test):.3f}")

**The coefficients.** The feature names have to come out of the preprocessor, because
encoding created columns that were not in the original table.

In [ ]:
names = toyota_model.named_steps['preprocessor'].get_feature_names_out()
coefs = toyota_model.named_steps['regressor'].coef_

pd.DataFrame({'feature': names, 'coefficient': coefs})

---
## Steps to Train a Model

Five steps, in this order, for every model this term. Photograph it.

1. **What am I predicting?** That column is `y`. Everything else is `X`.
2. **Split first** — before you touch a single column.
3. **Which columns are already numbers?** Those go in as they are.
4. **Which ones are not?** Those get encoded.
5. **Fit on training. Score on both.** A number from the training set alone tells you nothing.

Step 2 is the one that costs people their jobs. It is the **preprocessing leakage** promised in
Week 3: prepare the data before you split, and the test set has already leaked into the model.

---
### Before you close this

Parts **1c**, **1d**, **2b**, **2c** and **2d** are sentences, not code. They are the part
an AI cannot do for you, because every one of them depends on what a column is measured in and what
the business question was. They are also the part that gets read.

This week's homework picks up from section 3, and asks you to build the Toyota model yourself.